In [1]:
"""
fin문철&test
"""

import os
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from typing import Dict, List

from langchain_naver import ChatClovaX
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage

import dotenv
dotenv.load_dotenv()

True

In [ ]:


class StockDataFetcher:
    """주식 데이터 수집 클래스"""
    
    @staticmethod
    def get_stock_info(symbol: str) -> Dict:
        """주식 기본 정보 조회"""
        try:
            # 한국 주식의 경우 .KS 추가
            if not '.' in symbol:
                symbol = f"{symbol}.KS"
            
            stock = yf.Ticker(symbol)
            info = stock.info
            
            # 최근 1년 데이터 조회
            hist = stock.history(period="1y")
            
            # 기술적 지표 계산
            current_price = hist['Close'][-1]
            sma_20 = hist['Close'].rolling(window=20).mean()[-1]
            sma_60 = hist['Close'].rolling(window=60).mean()[-1]
            
            # RSI 계산 (간단 버전)
            delta = hist['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            rsi = 100 - (100 / (1 + rs))
            
            return {
                'symbol': symbol,
                'company_name': info.get('longName', 'N/A'),
                'current_price': current_price,
                'market_cap': info.get('marketCap', 0),
                'pe_ratio': info.get('trailingPE', 0),
                'pb_ratio': info.get('priceToBook', 0),
                'dividend_yield': info.get('dividendYield', 0),
                'sma_20': sma_20,
                'sma_60': sma_60,
                'rsi': rsi[-1] if len(rsi) > 0 else 50,
                'volume': hist['Volume'][-1],
                'price_change_1d': ((current_price - hist['Close'][-2]) / hist['Close'][-2] * 100),
                'price_change_1w': ((current_price - hist['Close'][-5]) / hist['Close'][-5] * 100),
                'price_change_1m': ((current_price - hist['Close'][-21]) / hist['Close'][-21] * 100),
                'history': hist
            }
        except Exception as e:
            print(f"주식 데이터 조회 오류: {e}")
            return None


class InvestmentAnalyzer:
    """투자 분석 클래스"""
    
    def __init__(self):
        # Clova X 모델 초기화
        self.llm = ChatClovaX(
            api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
            model="HCX-005",
            temperature=0.3,
            max_tokens=3000,
        )
        
        self.analysis_prompt = PromptTemplate(
            input_variables=["stock_data", "user_query", "analysis_type"],
            template="""
당신은 전문적인 투자 분석가 "Fin문철"입니다. 사용자의 투자 판단을 블랙박스처럼 분해하여 객관적으로 분석해주세요.

**주식 데이터:**
{stock_data}

**사용자 질문:** {user_query}

**분석 유형:** {analysis_type}

다음 관점에서 종합적으로 분석해주세요:

**📊 정량적 분석 (Quantitative Analysis)**
1. 기술적 지표 분석 (RSI, 이동평균선 등)
2. 재무 지표 분석 (PER, PBR, 배당수익률 등)
3. 가격 변동성 및 거래량 분석

**📈 정성적 분석 (Qualitative Analysis)**
1. 현재 시장 상황 및 업종 전망
2. 회사 기본면 및 경쟁력
3. 리스크 요인 및 기회 요소

**🎯 투자 판단 종합 평가**
- 현재 가격의 적정성
- 투자 타이밍 평가
- 추천 투자 전략
- 위험도 평가 (1-10점)

**📋 블랙박스 분석 결과**
최종 평가: [매우 좋음/좋음/보통/주의/위험] 중 하나로 판단하고 근거를 명확히 제시해주세요.

투자는 본인의 판단과 책임입니다. 이 분석은 참고용이며 투자 권유가 아닙니다.
            """
        )
    
    def analyze_investment(self, stock_data: Dict, user_query: str, analysis_type: str = "종합분석") -> str:
        """투자 분석 수행"""
        try:
            # 주식 데이터를 텍스트로 변환
            stock_summary = f"""
종목명: {stock_data['company_name']} ({stock_data['symbol']})
현재가: {stock_data['current_price']:,.0f}원
시가총액: {stock_data['market_cap']:,}
PER: {stock_data['pe_ratio']:.2f}
PBR: {stock_data['pb_ratio']:.2f}
배당수익률: {stock_data['dividend_yield']*100:.2f}%
RSI: {stock_data['rsi']:.1f}
20일 이평: {stock_data['sma_20']:,.0f}원
60일 이평: {stock_data['sma_60']:,.0f}원
1일 변동률: {stock_data['price_change_1d']:+.2f}%
1주 변동률: {stock_data['price_change_1w']:+.2f}%
1개월 변동률: {stock_data['price_change_1m']:+.2f}%
거래량: {stock_data['volume']:,}주
            """
            
            prompt = self.analysis_prompt.format(
                stock_data=stock_summary,
                user_query=user_query,
                analysis_type=analysis_type
            )
            
            response = self.llm.invoke([HumanMessage(content=prompt)])
            return response.content
            
        except Exception as e:
            return f"분석 중 오류 발생: {e}"


class ScenarioSimulator:
    """시나리오 시뮬레이션 클래스"""
    
    @staticmethod
    def simulate_scenarios(stock_data: Dict, scenarios: List[Dict]) -> pd.DataFrame:
        """다양한 시나리오 하에서 주가 영향 시뮬레이션"""
        current_price = stock_data['current_price']
        results = []
        
        for scenario in scenarios:
            name = scenario['name']
            impact = scenario['impact']  # -0.2 ~ +0.3 등
            
            new_price = current_price * (1 + impact)
            change_pct = impact * 100
            
            results.append({
                '시나리오': name,
                '예상 주가': f"{new_price:,.0f}원",
                '변동률': f"{change_pct:+.1f}%",
                '영향도': '긍정적 🔺' if impact > 0 else '부정적 🔻' if impact < 0 else '중립적'
            })
        
        return pd.DataFrame(results)


def create_price_chart(stock_data: Dict) -> go.Figure:
    """주가 차트 생성"""
    hist = stock_data['history']
    
    fig = go.Figure()
    
    # 캔들스틱 차트
    fig.add_trace(go.Candlestick(
        x=hist.index,
        open=hist['Open'],
        high=hist['High'],
        low=hist['Low'],
        close=hist['Close'],
        name='주가'
    ))
    
    # 이동평균선
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['Close'].rolling(window=20).mean(),
        name='20일 이평선',
        line=dict(color='orange', width=2)
    ))
    
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['Close'].rolling(window=60).mean(),
        name='60일 이평선',
        line=dict(color='blue', width=2)
    ))
    
    fig.update_layout(
        title=f"{stock_data['company_name']} 주가 차트",
        yaxis_title="주가 (원)",
        xaxis_title="날짜",
        height=400,
        xaxis_rangeslider_visible=False
    )
    
    return fig

In [ ]:
"""
Extracted Classes from Streamlit Applications
"""

import os
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from typing import Dict, List, Optional
from langchain_naver import ChatClovaX
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage
import dotenv

dotenv.load_dotenv()


class FinancialCalculator:
    """금융 계산 클래스"""
    
    @staticmethod
    def compound_interest(principal, rate, time):
        """복리 계산: A = P(1 + r)^t"""
        return principal * (1 + rate) ** time
    
    @staticmethod
    def simple_interest(principal, rate, time):
        """단리 계산: A = P(1 + rt)"""
        return principal * (1 + rate * time)
    
    @staticmethod
    def equal_payment_loan(principal, rate, time_years):
        """원리금균등상환 월납부액"""
        if rate == 0:
            return principal / (time_years * 12)
        
        monthly_rate = rate / 12
        n_payments = time_years * 12
        
        payment = principal * (monthly_rate * (1 + monthly_rate) ** n_payments) / \
                 ((1 + monthly_rate) ** n_payments - 1)
        return payment
    
    @staticmethod
    def bullet_payment_interest(principal, rate, time_years):
        """만기일시상환 총이자"""
        return principal * rate * time_years
    
    @staticmethod
    def max_loan_amount(monthly_payment, rate, time_years):
        """월납부액으로 가능한 최대 대출금액"""
        if rate == 0:
            return monthly_payment * time_years * 12
        
        monthly_rate = rate / 12
        n_payments = time_years * 12
        
        principal = monthly_payment * ((1 + monthly_rate) ** n_payments - 1) / \
                   (monthly_rate * (1 + monthly_rate) ** n_payments)
        return principal


class FinancialQueryProcessor:
    """자연어 금융 질의 처리 클래스"""
    
    def __init__(self):
        self.llm = ChatClovaX(
            api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
            model="HCX-005",
            temperature=0.2,
            max_tokens=2000,
        )
        
        self.prompt_template = PromptTemplate(
            input_variables=["query"],
            template="""
당신은 금융 계산 전문가입니다. 사용자의 질의를 분석하여 정확한 금융 계산을 수행하고 결과를 설명해주세요.

다음 공식들을 사용하여 계산하세요:

1. 복리 계산: A = P(1 + r)^t
   - A: 최종 금액, P: 원금, r: 연이자율, t: 기간(년)

2. 원리금균등상환 월납부액: PMT = P × [r(1+r)^n] / [(1+r)^n - 1]
   - PMT: 월납부액, P: 대출원금, r: 월이자율, n: 총 납부횟수

3. 만기일시상환 이자: I = P × r × t
   - I: 총 이자, P: 원금, r: 연이자율, t: 기간

4. 최대 대출액: P = PMT × [(1+r)^n - 1] / [r(1+r)^n]

사용자 질의: {query}

계산 과정과 결과를 명확하게 설명해주세요. 
단위 변환 (억=100,000,000원, 만=10,000원)도 적용하세요.
            """
        )
    
    def process_query(self, query: str) -> str:
        """자연어 질의 처리"""
        try:
            prompt = self.prompt_template.format(query=query)
            response = self.llm.invoke([HumanMessage(content=prompt)])
            return response.content
        except Exception as e:
            return f"죄송합니다. 처리 중 오류가 발생했습니다: {e}"


class StockDataFetcher:
    """주식 데이터 수집 클래스"""
    
    @staticmethod
    def get_stock_info(symbol: str) -> Dict:
        """주식 기본 정보 조회"""
        try:
            if not '.' in symbol:
                symbol = f"{symbol}.KS"
            
            stock = yf.Ticker(symbol)
            info = stock.info
            hist = stock.history(period="1y")
            
            current_price = hist['Close'][-1]
            sma_20 = hist['Close'].rolling(window=20).mean()[-1]
            sma_60 = hist['Close'].rolling(window=60).mean()[-1]
            
            delta = hist['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            rsi = 100 - (100 / (1 + rs))
            
            return {
                'symbol': symbol,
                'company_name': info.get('longName', 'N/A'),
                'current_price': current_price,
                'market_cap': info.get('marketCap', 0),
                'pe_ratio': info.get('trailingPE', 0),
                'pb_ratio': info.get('priceToBook', 0),
                'dividend_yield': info.get('dividendYield', 0),
                'sma_20': sma_20,
                'sma_60': sma_60,
                'rsi': rsi[-1] if len(rsi) > 0 else 50,
                'volume': hist['Volume'][-1],
                'price_change_1d': ((current_price - hist['Close'][-2]) / hist['Close'][-2] * 100),
                'price_change_1w': ((current_price - hist['Close'][-5]) / hist['Close'][-5] * 100),
                'price_change_1m': ((current_price - hist['Close'][-21]) / hist['Close'][-21] * 100),
                'history': hist
            }
        except Exception as e:
            print(f"주식 데이터 조회 오류: {e}")
            return None


class InvestmentAnalyzer:
    """투자 분석 클래스"""
    
    def __init__(self):
        self.llm = ChatClovaX(
            api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
            model="HCX-005",
            temperature=0.3,
            max_tokens=3000,
        )
        
        self.analysis_prompt = PromptTemplate(
            input_variables=["stock_data", "user_query", "analysis_type"],
            template="""
당신은 전문적인 투자 분석가 "Fin문철"입니다. 사용자의 투자 판단을 블랙박스처럼 분해하여 객관적으로 분석해주세요.

**주식 데이터:**
{stock_data}

**사용자 질문:** {user_query}

**분석 유형:** {analysis_type}

다음 관점에서 종합적으로 분석해주세요:

**📊 정량적 분석 (Quantitative Analysis)**
1. 기술적 지표 분석 (RSI, 이동평균선 등)
2. 재무 지표 분석 (PER, PBR, 배당수익률 등)
3. 가격 변동성 및 거래량 분석

**📈 정성적 분석 (Qualitative Analysis)**
1. 현재 시장 상황 및 업종 전망
2. 회사 기본면 및 경쟁력
3. 리스크 요인 및 기회 요소

**🎯 투자 판단 종합 평가**
- 현재 가격의 적정성
- 투자 타이밍 평가
- 추천 투자 전략
- 위험도 평가 (1-10점)

**📋 블랙박스 분석 결과**
최종 평가: [매우 좋음/좋음/보통/주의/위험] 중 하나로 판단하고 근거를 명확히 제시해주세요.

투자는 본인의 판단과 책임입니다. 이 분석은 참고용이며 투자 권유가 아닙니다.
            """
        )
    
    def analyze_investment(self, stock_data: Dict, user_query: str, analysis_type: str = "종합분석") -> str:
        """투자 분석 수행"""
        try:
            stock_summary = f"""
종목명: {stock_data['company_name']} ({stock_data['symbol']})
현재가: {stock_data['current_price']:,.0f}원
시가총액: {stock_data['market_cap']:,}
PER: {stock_data['pe_ratio']:.2f}
PBR: {stock_data['pb_ratio']:.2f}
배당수익률: {stock_data['dividend_yield']*100:.2f}%
RSI: {stock_data['rsi']:.1f}
20일 이평: {stock_data['sma_20']:,.0f}원
60일 이평: {stock_data['sma_60']:,.0f}원
1일 변동률: {stock_data['price_change_1d']:+.2f}%
1주 변동률: {stock_data['price_change_1w']:+.2f}%
1개월 변동률: {stock_data['price_change_1m']:+.2f}%
거래량: {stock_data['volume']:,}주
            """
            
            prompt = self.analysis_prompt.format(
                stock_data=stock_summary,
                user_query=user_query,
                analysis_type=analysis_type
            )
            
            response = self.llm.invoke([HumanMessage(content=prompt)])
            return response.content
            
        except Exception as e:
            return f"분석 중 오류 발생: {e}"


class ScenarioSimulator:
    """시나리오 시뮬레이션 클래스"""
    
    @staticmethod
    def simulate_scenarios(stock_data: Dict, scenarios: List[Dict]) -> pd.DataFrame:
        """다양한 시나리오 하에서 주가 영향 시뮬레이션"""
        current_price = stock_data['current_price']
        results = []
        
        for scenario in scenarios:
            name = scenario['name']
            impact = scenario['impact']
            
            new_price = current_price * (1 + impact)
            change_pct = impact * 100
            
            results.append({
                '시나리오': name,
                '예상 주가': f"{new_price:,.0f}원",
                '변동률': f"{change_pct:+.1f}%",
                '영향도': '긍정적 🔺' if impact > 0 else '부정적 🔻' if impact < 0 else '중립적'
            })
        
        return pd.DataFrame(results)


def create_comparison_chart(results: List[Dict]) -> go.Figure:
    """비교 차트 생성"""
    fig = go.Figure()
    
    periods = [r['period'] for r in results]
    amounts = [r['total_amount'] for r in results]
    interests = [r['total_interest'] for r in results]
    
    fig.add_trace(go.Bar(
        name='총 상환액',
        x=periods,
        y=amounts,
        marker_color='lightblue'
    ))
    
    fig.add_trace(go.Bar(
        name='총 이자',
        x=periods,
        y=interests,
        marker_color='coral'
    ))
    
    fig.update_layout(
        title='대출 기간별 비교',
        xaxis_title='기간',
        yaxis_title='금액 (원)',
        barmode='group',
        height=400
    )
    
    return fig


def create_price_chart(stock_data: Dict) -> go.Figure:
    """주가 차트 생성"""
    hist = stock_data['history']
    
    fig = go.Figure()
    
    fig.add_trace(go.Candlestick(
        x=hist.index,
        open=hist['Open'],
        high=hist['High'],
        low=hist['Low'],
        close=hist['Close'],
        name='주가'
    ))
    
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['Close'].rolling(window=20).mean(),
        name='20일 이평선',
        line=dict(color='orange', width=2)
    ))
    
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['Close'].rolling(window=60).mean(),
        name='60일 이평선',
        line=dict(color='blue', width=2)
    ))
    
    fig.update_layout(
        title=f"{stock_data['company_name']} 주가 차트",
        yaxis_title="주가 (원)",
        xaxis_title="날짜",
        height=400,
        xaxis_rangeslider_visible=False
    )
    
    return fig